# Manuscript figures

The figures being prepared for the paper, in the order they are meant to appear.  Each one is drawn by the analysis
library that owns it (`lib/uniaxial.py`, `lib/compare.py`, `lib/chain_length.py`; two are drawn in `lib/manuscript.py` itself) and then finished here by
`lib/manuscript.py`, so the text, fonts, size, limits and legend of every figure can be changed in its own cell without
touching the library.

**How to edit a figure.**  Each figure cell starts with a `TEXT` dict that lists every piece of text in the figure by key
(`ax0.xlabel`, `ax0.legend.2`, `ax1.text.0`, ...).  Change a string to change the text; `''` removes it; `None` keeps what
the library writes.  Text that carries a run name or a fitted number is listed as `None` with its current wording in a
comment, so it follows the data until you replace it.  `ms.print_text(fig)` prints the keys of any figure.  Below `TEXT`:
`FONTS` (sizes in points), `SIZE` (inches), `LIMITS`, `LEGENDS`, and `STYLE` for per-figure overrides of the global settings.

**Global settings** are in the second code cell (`ms.set_style`): font, sizes, line widths, output folder and formats.

Order: (1) gel volume over the equilibration, (2) strain and solvent exchange over the compression cycle, (3) solvent
and polymer partial-stress profiles across the compression cell, (4) displacement evolution in re-swelling and in
permeation, (5) the networks and their frozen-pore fraction.  The earlier cross-deck comparison figures (D_c and κ
against φ_s) follow at the end; they need a live-network permeation run and are skipped while none is loaded.
This notebook was `cross_deck_comparison.ipynb` until 2026-11-22.

## Setup

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: uniaxial.py, compare.py, chain_length.py (the figures) + manuscript.py (the finishing)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import uniaxial as uni
import compare as cmp
import chain_length as cl
import manuscript as ms
uni = importlib.reload(uni); cmp = importlib.reload(cmp); cl = importlib.reload(cl); ms = importlib.reload(ms)
uni.setup_style()
print('figure finishing: ', LIB / 'manuscript.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  GLOBAL FIGURE SETTINGS -- apply to every figure below; a figure cell's STYLE = {...} overrides them for that figure
# ══════════════════════════════════════════════════════════════════════════
ms.set_style(
    font_family      = None,      # e.g. 'CMU Serif' (the notebooks' font), 'Times New Roman', 'Helvetica', 'Arial'; None = leave
    mathtext_fontset = None,      # 'cm' | 'stix' | 'dejavusans' | 'dejavuserif'; None = leave
    font_scale       = 1.0,       # multiplies every font size the library used
    size_suptitle    = None,      # fixed sizes in points per kind of text (None = library size x font_scale)
    size_title       = None,
    size_label       = None,
    size_tick        = None,
    size_legend      = None,
    size_text        = None,      # annotations, text boxes, notes
    lw_scale         = 1.0,       # multiplies every curve's line width
    ms_scale         = 1.0,       # multiplies every marker size
    spine_lw         = None,      # axes frame line width
    tick_direction   = None,      # 'in' | 'out' | 'inout'
    grid             = None,      # True / False forces the grid on / off everywhere; None = as drawn
    suptitle         = True,      # False drops every figure title (the caption carries it in a paper)
    panel_titles     = True,      # False drops every panel title that the cell's TEXT does not set
    dpi              = 300,
    formats          = ('png', 'pdf'),
    transparent      = False,
    out_dir          = Path('../../flow_data_local/plots/manuscript'),   # None = do not save
);

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- one Config per deck, copied from that deck's own notebook, + the comparison knobs
# ══════════════════════════════════════════════════════════════════════════
CFG = {}

# LOADED (2026-11-22): the random 10-bead wide gel -- one compression level with its unload.  No permeation run (see below).
# Previous set (diamond-lattice 5-bead gel; DATANAME final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist[_pm40]):
#   compression periodic_rho04_14M_9M_twoPlatesMove_7_sweep (levels 0.10-0.5), permeation periodic_rho04_14M_11M_perm_3,
#   unload two_pist_rho04_14M_10M_25M_comp_4_andrelax (level 0.10); ccfg NAME comp7_perm3, PERM_FROZEN_REF False.

# ---- compression: the Config cell of uniaxial_compression_single.ipynb.  This gel has no sweep yet, so the one level (0.10) is
#      the whole equation of state (reference + one point) AND, through its unload, the compression-side D_c ----
CFG['compression'] = uni.Config(
    DATANAME    = "final_config_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_16000000_two_pist_pm40",
    INTERACTION = "1.0_1.0",
    NSTEPS      = None,
    RUN_ID      = "random_16M_comp_1",        # job 55793779 (2026-11-17, SEAT_MODE=rg): hold 4 M + full unload
    COMP_LEVELS = ["0.10"],
    mode        = "compression",
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
    M_SUBTRACT_REF = True, G_SUBTRACT_REF = True,
    DC_FREE_AMPS = True, DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
    DC_PRIMARY = 'unload',
)

# ---- permeation: NOT LOADED (2026-11-22).  The only permeation run of this gel so far, random_10b_wide_perm_fref_1, has its
#      network frozen undeformed for the whole drive.  A frozen network's stress is one instant's value, never time-averaged,
#      so its stress profiles are ~50x rougher from bin to bin than a live run's (0.077 against 0.0014 in total stress, for a
#      pressure drop of 0.1), and it has no displacement to fit.  Put the live-network run here when it has been pulled
#      (uniaxial_permeation_single.ipynb, its Config cell): set RUN_ID and NSTEPS, uncomment, and set PERM_FROZEN_REF = False
#      below.  The frozen-reference Config is kept, commented, for the Darcy kappa comparison.
# # ---- permeation: the Config cell of uniaxial_permeation_single.ipynb (+ the figure knobs below).  This run's network is FROZEN
# #      undeformed for the whole drive (deck FROZEN_AT=ref): it gives the zero-strain Darcy kappa only (PERM_FROZEN_REF below) ----
# CFG['permeation'] = uni.Config(
#     DATANAME    = "final_config_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_16000000_two_pist_pm40",
#     INTERACTION = "1.0_1.0",
#     NSTEPS      = 4000000,            # pinned: a live drive of this gel will share the DATANAME
#     RUN_ID      = "random_10b_wide_perm_fref_1",   # job 55745208 (2026-11-15/16), dP 0.1, 4 M steps
#     mode        = "permeation",
#     DP_PISTON   = None,
#     plateau_frac      = 0.25,
#     plateau_frac_auto = 0.45,
#     P_BARO      = 1.5,
#     VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, VOR_EVO_FRAMES = 5,
#     P_CAL = 1.5, P_CAL_MODE = 'pore',
#     G_REF = None, G_COMP_REF = None,   # the two dP_th = (4/3)(G/M) dP_ext guides of the normalised P_th figure: none for this gel yet
#                                        # (lattice gel: 0.2 / 0.05); a frozen network has no M to draw them with anyway
#     PSD_ENABLE = True, PSD_R_PROBE = 0.45, PSD_GRID = 0.5,   # probe porosity for the solvent-measures figure.  r_probe 0.45 < the permeation
#                                        # notebook's 0.5: the probe centre must clear a polymer bead centre by sigma_bead/2 + r_probe = 0.95 sigma,
#                                        # about where U_LJ = kT (0.97 sigma) -- the distance a thermal solvent bead actually reaches
#     PERM_GAP = 1.0, PERM_L0_MODE = 'face', PERM_L0_STEPS = 500000, PERM_DC_WINDOW_AVG = True, PERM_DISP_RESET = 'auto', PERM_TRACE_SKIP = 1500000,
#     PERM_DC_BOUNDS = (1e-6, 50.0), PERM_RIGID = 'contact', PERM_PIN_NBINS = 2, PERM_COORDS = 'lagrangian',
#     PERM_DC_N_MODES = 8, PERM_M_PRIMARY = 'lag',
#     DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_FRAC_EARLY = 1.0,
# )

# ---- unload: a SEPARATE compression run whose levels have _u<lvl> files (as the lattice gel's _andrelax run was).  Not needed
#      here: the compression run above carries its own unload, and every level of CFG['compression'] with one is picked up.
#      Add CFG['unload'] = uni.Config(...) only when the unload comes from a different run ----

# ---- another deck: add its Config here and it is loaded with the others (see "Adding a deck" in the Notes) ----
# import shear as sh
# CFG['shear'] = sh.Config(
#     DATANAME = "isolated_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_with_plates", INTERACTION = "1.0_1.0",
#     NSTEPS = None, RUN_ID = "periodic_rho04_14M_shear_1", STRAINS = ["0.1", "0.15", "0.2", "0.25", "0.3"],
#     plateau_frac = 0.25, plateau_frac_auto = 0.45, G_SUBTRACT_REF = True,
#     DC_N_MODES = 5, DC_TRIM_BINS = 1, DC_SLOW_REF = 0.03, DC_TARGET_RESID = 0.01,
# )

# ---- comparison knobs (cmp.Config in lib/compare.py) ----
ccfg = cmp.Config(
    NAME          = "random10b_comp1",             # plots -> flow_data_local/plots/comparison/<NAME>
    PERM_FROZEN_REF = True,           # the loaded permeation run's network is frozen undeformed: no local D_c, M_perm or closures -- the Darcy
                                      # kappa is compared with the unload kappa = D_c/M.  False for a live-network permeation run
    PHI_KIND      = 'cal',            # phi_s used throughout: 'cal' (lambda-calibrated Voronoi) | 'vor' (Voronoi) | 'mf' (mass fraction)
    PERM_TRIM     = 6.0,              # sigma dropped at each end of the membrane before the phi_s(z) slope is taken
    PERM_N_SEG    = 3,                # z-segments of the steady profile, one local D_c / kappa each
    PERM_POLY_DEG = 2,                # degree of the polynomial phi_s(z) behind the continuous curves
    CONC_MARGIN   = 5.0,              # solvent-measures figure: sigma of reservoir kept on either side of the membrane (the rest is trimmed)
    CONC_FRAMES   = True,             # ... draw each steady Voronoi frame of c_s faintly (the noise under the band)
    CONC_DIFF_PANEL = True,           # ... panel (b): mass fraction and eps_g minus the calibrated fraction
)

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    for name, c in CFG.items():
        cmp.sync_deck(c, force=FORCE_SYNC)

In [ ]:
# DECKS[name] = dict(kind, cfg, R, levels | P) -- each run loaded by its own library (Voronoi profiles come from the disk
# cache the source notebooks fill; a first load of a new run tessellates ~20 s per frame).  The permeation deck also gets the
# probe-porosity pass (PSD_ENABLE; ~7 s per frame, re-streams the trajectory frames) for the solvent-measures figure.
DECKS = {name: cmp.load_deck(c, verbose=False) for name, c in CFG.items()}
HAVE_PERM = 'permeation' in DECKS          # the cross-deck comparison cells at the end run only with a permeation deck

## Figure 1 · Gel volume over the equilibration

The two gel-volume curves that the postprocessing script (`scripts/postprocess.sh`, `lib/plot_lammps_log.py`) draws for
the equilibration of the periodic random 10-bead network (`slab_support_periodic_10beads_wide_rho04_random`,
`slab_with_support` deck, 2026-11-15, 14.6 M steps = 73,000 τ): the volume from the polymer's radius of gyration,
`R_g³`, and the volume of the polymer's bounding box (BB), each divided by its first sample and written every 10 k
steps.  The dotted line is the step at which the deck's `R_g` halt detected the plateau (three consecutive 1 M-step
blocks with the block-mean `R_g` changing by less than 0.15 %); the run then continued for two more blocks.  The BB
volume is set by the outermost beads and is the noisier of the two.  `LAYOUT = 'stacked'` gives each curve its own panel,
as in the postprocessing figure.  This figure does not need the decks loaded.

In [ ]:
# ── Figure 1 ─────────────────────────────────────────────────────────────────────────────
EQ_RUN  = '*_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_*'   # run directory under flow_data_local/raw_data/chain_length (newest match wins)
LAYOUT  = 'single'       # 'single' (both curves on one panel) | 'stacked' (one panel each)
TAIL    = 0.3            # fraction of the run, at its end, behind the quoted means (the postprocessing script uses the last 30 %)
PLATEAU = True           # mark the step at which the Rg halt detected the plateau
TEXT = {
    'suptitle': None,   # now: 'Gel volume over the equilibration  |  chain_length_testing_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_20261115_035450'
    'ax0.xlabel': r"time  ($10^3\,\tau$)",
    'ax0.ylabel': 'gel volume / initial',
    'ax0.legend.0': r"$R_g^{\,3}$ volume",
    'ax0.legend.1': 'bounding-box (BB) volume',
    'ax0.legend.2': '$R_g$ plateau detected',
    'ax0.text.0': None,   # now: ('mean of the last 30 %' '\n' r"$R_g^{\,3}$ volume: 0.8622" '\n' 'bounding-box (BB) volume: 0.9738')
}
FONTS   = dict(suptitle=None, title=None, label=None, tick=None, legend=None, text=None)   # points; None = the global setting.  One piece: FONTS['ax0.xlabel'] = 12
SIZE    = None      # (width, height) in inches, e.g. (7.0, 4.3); None = as drawn
LIMITS  = {}        # {'ax0': dict(xlim=(0, 60000), ylim=(-0.25, 0.25), xticks=[...], xscale='log')}
LEGENDS = {}        # {'ax0': dict(loc='upper right', ncol=1, frameon=False)};  {'ax0': False} removes it
STYLE   = {}        # global settings overridden for this figure only, e.g. dict(suptitle=False, lw_scale=1.2)

GV = ms.load_gel_volume(EQ_RUN)
with ms.capture():
    fig = ms.fig_gel_volume(GV, layout=LAYOUT, tail=TAIL, plateau=PLATEAU)
ms.finish(fig, name='fig1_gel_volume', text=TEXT, size=SIZE, fonts=FONTS, limits=LIMITS, legends=LEGENDS, style=STYLE);

## Figure 2 · Strain and solvent exchange over the compression cycle

One level of the uniaxial compression run on one time axis: load (ramp, then hold at fixed plates) and unload (plates
retract, then free re-swelling).  Left axis: the strains.  Right axis: the rate at which solvent enters (> 0) or leaves
(< 0) the gel, defined as the slab holding the central 95 % of the network mass (`SOLV_Q` in the compression Config).
The two zero lines coincide.  `LAYOUT = 'stacked'` draws the two as separate panels sharing the time axis.

In [ ]:
# ── Figure 2 ─────────────────────────────────────────────────────────────────────────────
LAYOUT = 'twin'        # 'twin' (one panel, two y axes) | 'stacked' (two panels)
TEXT = {
    'suptitle': None,   # now: ('Strain (left) and solvent exchange (right) over the load and unload of level _c0.10  |  final_config_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_16000000_two_pist_pm40_1.0_1.0_4000000' '\n' r"ramp 200 $\tau$;  retraction 600 $\tau$ (the two darker bands);  the two zero lines coincide;  right axis linear inside $\pm$1.0, logarithmic outside")
    'ax0.xlabel': r"time since the ramp started  ($\tau$)",
    'ax0.ylabel': r"strain  $\varepsilon=(L_0-L)/L_0$",
    'ax0.legend.0': r"$\varepsilon_{Rg}$: network, $R_g$ thickness",
    'ax0.legend.1': r"$\varepsilon_{BB}$: network, outermost beads",
    'ax0.legend.2': r"$\varepsilon_\mathrm{plates}$: plate gap",
    'ax0.legend.3': 'right axis: solvent entering ($>0$) / leaving ($<0$) the central 95 % of the network mass',
    'ax0.legend.4': 'ramp / retraction: no profiles written, interpolated between the bracketing values',
    'ax0.text.0': 'LOAD: hold at fixed plates',
    'ax0.text.1': 'UNLOAD: free re-swelling',
    'ax0.text.2': None,   # now: 'end of hold: Rg 0.0863  BB 0.100'
    'ax0.text.3': None,   # now: 'end of unload: Rg -0.0557  BB -0.174'
    'ax1.ylabel': r"solvent entering the gel  (beads/$\tau$)",
}
FONTS   = dict(suptitle=None, title=None, label=None, tick=None, legend=None, text=None)   # points; None = the global setting.  One piece: FONTS['ax0.xlabel'] = 12
SIZE    = None      # (width, height) in inches, e.g. (7.0, 4.3); None = as drawn
LIMITS  = {}        # {'ax0': dict(xlim=(0, 60000), ylim=(-0.25, 0.25), xticks=[...], xscale='log')}
LEGENDS = {}        # {'ax0': dict(loc='upper right', ncol=1, frameon=False)};  {'ax0': False} removes it
STYLE   = {}        # global settings overridden for this figure only, e.g. dict(suptitle=False, lw_scale=1.2)

with ms.capture():
    fig = cmp.fig_strain_solvent(ccfg, DECKS['compression'], layout=LAYOUT)
ms.finish(fig, name='fig2_strain_solvent', text=TEXT, size=SIZE, fonts=FONTS, limits=LIMITS, legends=LEGENDS, style=STYLE);

## Figure 3 · Solvent and polymer partial stresses across the cell

The normal stress `σ_zz(z)` carried by the solvent, by the polymer network, and their sum, across the compression cell of
the random 10-bead gel (the gel equilibrated with the `slab_with_support` deck on 2026-11-15; compression run
`random_16M_comp_1`).  From left to right: feed piston, feed reservoir, load plate, gel, support plate, permeate
reservoir, permeate piston.  The reservoirs are tinted blue and the gel grey; plates and pistons are drawn where they
stand at the end of the phase.  Dashed curves are the uncompressed reference; the pale thin curves are earlier
profiles of the phase and the bold one is its last.  `PHASES = ('hold',)` shows the hold at fixed plates after the
ramp; add `'unload'` for a second panel with the free re-swelling after the plates retract.

In the reservoirs the solvent carries the whole stress (the bath pressure 1.5).  Inside the gel the polymer takes up
part of it and the solvent share drops by the same amount, so the total stays flat across the cell.  Compressing the
gel raises the polymer share (0.59 → 0.65) and lowers the solvent share.  The spike of the total at the feed piston is
the piston's own atoms.  Colours are the Okabe–Ito blue and vermilion with black for the total.

In [ ]:
# ── Figure 3 ─────────────────────────────────────────────────────────────────────────────
PHASES   = ('hold',)     # panels: 'hold' (reference -> compressed, plates fixed) and / or 'unload' (compressed -> re-swollen)
COMP     = 'zz'          # stress component: 'zz' | 'xx' | 'yy'
N_CURVES = None          # profiles drawn per species over the phase (None = the compression Config's n_curves)
TEXT = {
    'suptitle': None,   # now: (r"Partial and total $\sigma_{zz}$ profiles across the cell  |  final_config_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_16000000_two_pist_pm40_1.0_1.0_4000000")
    'ax0.title': None,   # now: r"hold at $\varepsilon=0.10$"
    'ax0.xlabel': r"position across the cell,  $1-z/L_z$",
    'ax0.ylabel': r"stress  $\sigma_{zz}$  (LJ)",
    'ax0.legend.0': r"solvent  $\sigma_{s,zz}$",
    'ax0.legend.1': r"polymer  $\sigma_{p,zz}$",
    'ax0.legend.2': r"total  $\sigma_{zz}$",
    'ax0.legend.3': r"reference ($\varepsilon=0$)",
    'ax0.legend.4': 'earlier in the hold (pale)',
    'ax0.legend.5': 'end of the hold (bold)',
    'ax0.text.0': 'feed\nreservoir',
    'ax0.text.1': 'gel',
    'ax0.text.2': 'permeate\nreservoir',
    'ax0.text.3': 'feed piston',
    'ax0.text.4': 'load plate',
    'ax0.text.5': 'support plate',
    'ax0.text.6': 'permeate piston',
}
# (with two PHASES the second panel adds the same keys under ax1: run ms.print_text(fig) after the cell to list them)
FONTS   = dict(suptitle=None, title=None, label=None, tick=None, legend=None, text=None)   # points; None = the global setting.  One piece: FONTS['ax0.xlabel'] = 12
SIZE    = None      # (width, height) in inches, e.g. (7.0, 4.3); None = as drawn
LIMITS  = {}        # {'ax0': dict(xlim=(0, 60000), ylim=(-0.25, 0.25), xticks=[...], xscale='log')}
LEGENDS = {}        # {'ax0': dict(loc='upper right', ncol=1, frameon=False)};  {'ax0': False} removes it
STYLE   = {}        # global settings overridden for this figure only, e.g. dict(suptitle=False, lw_scale=1.2)

with ms.capture():
    fig = ms.fig_stress_profiles(DECKS['compression'], phases=PHASES, comp=COMP, n_curves=N_CURVES)
ms.finish(fig, name='fig3_stress_profiles', text=TEXT, size=SIZE, fonts=FONTS, limits=LIMITS, legends=LEGENDS, style=STYLE);

## Figure 4 · Displacement evolution: re-swelling after uniaxial compression, and permeation

(a) The free re-swelling of the compression level's unload: displacement profiles since the plates stopped (dots) and the
`cos kπζ` fit (lines), which gives the compression-side `D_c`.  (b) The permeation run's displacement profiles since the
pressure ramp and the consolidation fit (support face pinned, feed face free).  Panel (b) appears once a live-network
permeation run is loaded in the Config cell; `PERM_FIT = 'unload'` draws that run's own re-swelling instead of its drive.

**Which `D_c` from permeation.**  Nothing was fixed as *the* permeation `D_c`; four candidates exist in the code:

* the **transient profile fit** of panel (b), which uses the whole membrane, permeate-side layer included;
* the **steady local value** `D_c(φ_s) = q φ_p / (dφ_s/dz)` along the membrane (cross-deck figure 4 below), which can be
  read on the feed side alone;
* the **steady drop across the membrane**, `D_c = q L / ln[(1−φ_P)/(1−φ_F)]`, also per half: on the lattice gel the feed
  half gave 0.09 and the permeate half 0.15 (whole membrane 0.113);
* **`κ_Darcy × M`**, with `M = ΔP L_ss / (2|u_F|)` from the steady displacement profile (lattice gel: 0.096).

The compression-side reference is the re-swelling `D_c` of panel (a).

In [ ]:
# ── Figure 4 ─────────────────────────────────────────────────────────────────────────────
PERM_FIT = 'drive'     # panel (b): 'drive' (consolidation under the pressure drop) | 'unload' (the permeation run's re-swelling)
TEXT = {
    'suptitle': None,   # now: 'Displacement evolution and the poroelastic fit: re-swelling after uniaxial compression'
    'ax0.title': None,   # now: r"free re-swelling profiles ($\cos k\pi\zeta$, COM removed): $D_c=0.123$ over the first 6.0 $\tau_1$ (996 of 1426 snapshots, $R^2=0.997$), whole unload 0.127"
    'ax0.xlabel': r"$\zeta$ (material coordinate of the compressed gel)",
    'ax0.ylabel': '$u_z/L$ since the plates stopped',
    'ax1.ylabel': 'time step',
}
# (with a permeation deck loaded panel (b) adds its own keys: run ms.print_text(fig) after the cell to list them)
FONTS   = dict(suptitle=None, title=None, label=None, tick=None, legend=None, text=None)   # points; None = the global setting.  One piece: FONTS['ax0.xlabel'] = 12
SIZE    = None      # (width, height) in inches, e.g. (7.0, 4.3); None = as drawn
LIMITS  = {}        # {'ax0': dict(xlim=(0, 60000), ylim=(-0.25, 0.25), xticks=[...], xscale='log')}
LEGENDS = {}        # {'ax0': dict(loc='upper right', ncol=1, frameon=False)};  {'ax0': False} removes it
STYLE   = {}        # global settings overridden for this figure only, e.g. dict(suptitle=False, lw_scale=1.2)

with ms.capture():
    fig = ms.fig_displacement_evolution(DECKS['compression'], DECKS.get('permeation'), perm_fit=PERM_FIT)
ms.finish(fig, name='fig4_displacement_evolution', text=TEXT, size=SIZE, fonts=FONTS, limits=LIMITS, legends=LEGENDS, style=STYLE);

## Figure 5 · The four networks: bead fluctuations and the frozen-pore fraction

Top row: the mean network of each gel in a thin slab, every bead drawn as a disc whose radius is its rms in-plane
displacement (crosslinkers dark).  The box on each map gives the **frozen-pore fraction Φ** (`lib/screen_voids.py`, the
measure of the 2026-11 gel screen): a grid point is a pore in a frame when no polymer bead centre lies within the probe
radius `R_PROBE` of it, and Φ is the share of the pore pattern that stays in place over the trajectory window
(0 = pores are renewed everywhere, 1 = the same pores in every frame).  Here the window is the 16 frames (750 τ) of each
compression run's reference trajectory.  The random 5-bead gel keeps about a third of its pore pattern; the random
10-bead gel is at the level of the lattices.  Bottom row: the fluctuation budget, the bead localisation against length
scale, and the chord extension.

In [ ]:
# ── Figure 5 ─────────────────────────────────────────────────────────────────────────────
GELS = dict(cl.NETWORK_GELS)      # label -> glob of the gel's reference trajectory; remove or reorder lines to change the panels
R_PROBE = 1.0                     # probe radius of the frozen-pore fraction (sigma); 1.0 and 1.5 are computed
TEXT = {
    'suptitle': None,   # now: r"Network modes of the $\varepsilon = 0$ reference state: disc radius = rms in-plane displacement of the bead"
    'ax0.title': None,   # now: ('rot45, 5 beads' '\n' r"mean chord $\bar R = 2.32\,\sigma$,  4 bonds")
    'ax0.xlabel': r"$x$  ($\sigma$)",
    'ax0.ylabel': r"$z$  ($\sigma$)",
    'ax0.text.0': None,   # now: (r"frozen-pore fraction $\Phi$ = 0.05" '\n' r"(probe $r_p$ = 1 $\sigma$, 750 $\tau$)")
    'ax1.title': None,   # now: ('rot45, 15 beads' '\n' r"mean chord $\bar R = 4.98\,\sigma$,  14 bonds")
    'ax1.xlabel': r"$x$  ($\sigma$)",
    'ax1.ylabel': r"$z$  ($\sigma$)",
    'ax1.text.0': None,   # now: (r"frozen-pore fraction $\Phi$ = 0.04" '\n' r"(probe $r_p$ = 1 $\sigma$, 750 $\tau$)")
    'ax2.title': None,   # now: ('random, 5 beads' '\n' r"mean chord $\bar R = 2.72\,\sigma$,  4 bonds")
    'ax2.xlabel': r"$x$  ($\sigma$)",
    'ax2.ylabel': r"$z$  ($\sigma$)",
    'ax2.text.0': None,   # now: (r"frozen-pore fraction $\Phi$ = 0.31" '\n' r"(probe $r_p$ = 1 $\sigma$, 750 $\tau$)")
    'ax3.title': None,   # now: ('random, 10 beads' '\n' r"mean chord $\bar R = 4.19\,\sigma$,  9 bonds")
    'ax3.xlabel': r"$x$  ($\sigma$)",
    'ax3.ylabel': r"$z$  ($\sigma$)",
    'ax3.text.0': None,   # now: (r"frozen-pore fraction $\Phi$ = 0.07" '\n' r"(probe $r_p$ = 1 $\sigma$, 750 $\tau$)")
    'ax4.title': '(a) fluctuation budget',
    'ax4.ylabel': r"amplitude per axis / $\bar R^{\,2}$",
    'ax5.title': '(b) chain-bead localisation (dashed: mesh scale)',
    'ax5.xlabel': r"length scale  $2\pi/q$  (mean chords)",
    'ax5.ylabel': 'frozen fraction  $f_s$',
    'ax6.title': '(c) how taut the strands are',
    'ax6.xlabel': 'chord extension  $|R|/(nb)$',
    'ax6.ylabel': 'probability density',
    'ax7.legend.0': 'rot45, 5 beads',
    'ax7.legend.1': 'rot45, 15 beads',
    'ax7.legend.2': 'random, 5 beads',
    'ax7.legend.3': 'random, 10 beads',
    'ax7.legend.4': 'crosslinker cloud (maps)',
    'ax7.legend.5': 'chain-bead cloud (maps, gel colour)',
}
FONTS   = dict(suptitle=None, title=None, label=None, tick=None, legend=None, text=None)   # points; None = the global setting.  One piece: FONTS['ax0.xlabel'] = 12
SIZE    = None      # (width, height) in inches, e.g. (7.0, 4.3); None = as drawn
LIMITS  = {}        # {'ax0': dict(xlim=(0, 60000), ylim=(-0.25, 0.25), xticks=[...], xscale='log')}
LEGENDS = {}        # {'ax0': dict(loc='upper right', ncol=1, frameon=False)};  {'ax0': False} removes it
STYLE   = {}        # global settings overridden for this figure only, e.g. dict(suptitle=False, lw_scale=1.2)

TRAJ_DIR = CFG['compression'].TRAJ_DIR
NET_CACHE = globals().get('NET_CACHE', {})          # network modes kept per file, so a rerun only computes new gels
MODES = cl.load_network_modes(TRAJ_DIR, GELS, cache=NET_CACHE)
VOIDS = cl.load_void_persistence(TRAJ_DIR, GELS)    # cached as JSON next to the raw data
cl.print_void_table(VOIDS)
with ms.capture():
    fig = cl.fig_network_compare(MODES, voids=VOIDS, r_p=R_PROBE)
ms.finish(fig, name='fig5_networks_frozen_pores', text=TEXT, size=SIZE, fonts=FONTS, limits=LIMITS, legends=LEGENDS, style=STYLE);

## Earlier cross-deck comparison figures (need a permeation run)

The D_c(φ_s) and κ(φ_s) comparison of the former `cross_deck_comparison.ipynb`, unchanged.  Every cell is skipped while
no permeation deck is loaded (`HAVE_PERM`).  They are not yet wrapped for editing; wrap one as the figures above
(`with ms.capture(): fig = ...` then `ms.finish(fig, ...)`) when it is chosen for the paper.

In [ ]:
if HAVE_PERM:
    # 1 · Normalised thermodynamic pressure across the membrane (the permeation notebook's figure 10, lower panel): solvent, polymer
    #     and total traces with the back pressure removed, / dP_ext; reference dashed (~0).  Dash-dot guides: dP_th = (4/3)(G/M) dP_ext
    #     from the feed-reservoir P_th at the feed face down to the permeate side, one per G -- G_shear (G_REF, the shear decks) and
    #     G_comp (G_COMP_REF, the compression holds' lateral network stress), both with this run's M.  Saved under the deck's plot
    #     folder by its own library; a copy lands in the comparison folder.
    cmp.fig_thermo_pressure_norm(ccfg, DECKS['permeation']);
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # 2 · Three measures of the solvent across the steady membrane (the analogue of the permeation notebook's figure 7a), reference
    #     dashed / steady solid, 95 % bands: (a) concentration c_s = phi_s^cal rho_s,0/M_s (left axis; faint = each steady Voronoi
    #     frame), the mass (= number) fraction rho_s/rho_s,0, and the probe porosity eps_g (r_probe = PSD_R_PROBE) -- the fractions on
    #     the right axis, scaled by rho_s,0 so the three are directly comparable; reservoirs trimmed to CONC_MARGIN sigma.
    #     (b) the mass fraction and eps_g minus the calibrated fraction, bin by bin.
    cmp.fig_solvent_measures(ccfg, DECKS['permeation']);
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # X -- the phi_s comparison: S (compression states: phi_s, sigma', secant M -- no D_c), C (sigma'(phi_s) with CIs), PL (permeation,
    #      local), RW (every compression level with an UNLOAD, across the decks: its re-swelling D_c, secant M and kappa at its phi_s;
    #      None when no loaded level has one).  The hold D_c never enters (2026-10-21).
    X = cmp.phi_dependence(ccfg, DECKS['compression'], DECKS['permeation'], unload=DECKS.get('unload'))
    cmp.print_summary(ccfg, X)
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # 3 · The states: (a) steady permeation phi_s(z) with the z-segments, the polynomial and the compression levels' phi_s;
    #     (b) the compression curve sigma'(phi_s) with the applied dP at the membrane's support-side phi_s;
    #     (c) the current-frame tangent modulus M_cur = phi_p dsigma'/dphi_p, with the DISPLACEMENT-derived M of both decks: squares =
    #         the sweep levels' secant M = dsigma'/eps_disp (displacement-profile strain), band = the permeation M = dP L/(2|u_F|) from the
    #         steady free-face displacement, star = the re-swelling run's secant M (2026-10-15)
    cmp.fig_states(ccfg, X);
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # 4 · D_c(phi_s): the UNLOAD levels (free re-swelling profile fit, stars) -- the only
    #     compression-side D_c -- vs the steady permeation profile (local, q phi_p/(dphi_s/dz)); bars = the permeation notebook's
    #     membrane averages (transient fits, Darcy kappa x M) over the membrane's phi_s range.  No hold (load) D_c (2026-10-21).
    cmp.fig_Dc(ccfg, X);
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # 5 · kappa(phi_s): the UNLOAD levels' D_c / their secant M (stars) vs the local permeation kappa = D_c(phi_s)/M_perm;
    #     band = the Darcy membrane average Q L/(A dP).  No hold (load) kappa (2026-10-21).
    cmp.fig_kappa(ccfg, X);
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # 6 · Closures: (a) the network stress the compression curve assigns to the measured phi_s(z) against the linear pressure
    #     drop; (b) the measured flux against the one the UNLOAD D_c predicts for this profile (constant at the one unload level)
    cmp.fig_closure(ccfg, X);
else:
    print('skipped: no permeation deck loaded')

In [ ]:
if HAVE_PERM:
    # 7 · The same comparison under each phi_s definition (how much hangs on the choice, and on the Voronoi noise)
    cmp.print_phi_kinds(ccfg, DECKS['compression'], DECKS['permeation'], unload=DECKS.get('unload'))
else:
    print('skipped: no permeation deck loaded')

## Notes (cross-deck comparison)

* **Reading, random 10-bead wide gel (`random_16M_comp_1` + `random_10b_wide_perm_fref_1`; 2026-11-21).**  Unload `D_c` = 0.123
  (profile fit) and secant `M` = 0.197 give κ = D_c/M = 0.63 [0.60, 0.65] at
  φ_s 0.52.  The frozen-network permeation gives the Darcy κ = 0.57 [0.53, 0.62] at φ_s 0.59: ratio 1.09, and κ_Darcy × M = 0.113
  against the unload D_c 0.123.  The two states are not the same (the unload level is the more compressed one, where κ should be
  lower, not higher), and the frozen network removes the network's own motion from the flux, so this is a consistency check to
  ~10 %, not a like-for-like closure.  The frozen membrane's φ_s (0.59) is also above the compression run's ε = 0 reference
  (0.56): that reference is the seated state, and the Rg thickness shows the gel re-swelling past it in the unload (79.5 against
  74.4), i.e. the seated reference is itself a few percent compressed.

* **D_c sources: unload only (2026-10-21).**  Two kinds of D_c come out of a compression level.  *Load*: the hold after the
  ramp (`uni.fit_Dc` profile fit, the load-piston stress trace).  The hold starts from the ramp's face skins plus an undrained
  interior strain (the 2026-10-20 amplitude-spectrum analysis of run 7: only ~half the closure is hold-relaxing skin strain,
  the interior is already strained ~0.02 at the hold onset, the load piston spikes to ~0.3 during the ramp), and its D_c moves
  with the mode count, trim and window (0.05 from the decay rates, 0.065–0.085 from the mode spectrum).  *Unload*: the free
  re-swelling after the hold (deck Phase 2c, files `_u<lvl>`), which starts from a uniform state with free faces.  **Only the
  unload D_c enters this notebook**: `cmp.unload_states` collects every level with an unload across the compression decks in
  `CFG` (`CFG['unload']` is the fine-cadence `_andrelax` run, level 0.10, D_c 0.12), `cmp.compression_states` carries no D_c at
  all, closure (ii) and `print_phi_kinds` use the unload D_c (constant at one level, interpolated over several), and figures 4
  and 5 draw it as stars.  The sweep's hold D_c and κ are still in the sweep notebook, labelled LOAD.
* **What is compared.**  Both decks hold the gel laterally (fixed lx, ly) and strain it along z, so both visit the same
  one-parameter family of states, labelled by φ_s.  *Compression*: level i is a uniform state (φ_s,i, σ′_i) with the
  displacement-strain secant `M_i = Δσ′/ε_disp`; the levels give the equilibrium curve σ′(φ_s) (monotone PCHIP through the
  reference and the levels; CI from `N_MC` redraws of the levels' σ′ and φ_s) that the closures and figure 3(b) use.  No
  modulus comes from the volume fraction (the tangent `M_cur = φ_p dσ′/dφ_p` was dropped on 2026-10-15).  *Unload*: the
  level's re-swelling D_c with that level's secant `M`, `κ = D_c/M`.
* **Permeation, local.**  Steady state, network at rest: `q = κ |dp/dz|`, `dp/dz = dσ′/dz` (uniform total stress) and
  σ′ = σ′(φ_s) give `q = κ |dσ′/dφ_s| dφ_s/dz`, i.e. **`D_c(φ_s) = κ M_cur = q φ_p/(dφ_s/dz)`**.  It uses the N-slope
  flux and the slope of the steady φ_s(z) only.  `κ(φ_s) = D_c/M_cur` borrows `M_cur` from the compression curve, so the
  harmonic mean of the local κ over the membrane equals the Darcy `κ = QL/(AΔP)` exactly when closure (i) holds.
  The slope is taken per z-segment (straight-line fit, regression CI; a segment whose slope CI reaches 0 is drawn hollow
  with a lower bound only) and from one polynomial over the fit range (the continuous curve).
* **Which φ_s.**  `PHI_KIND = 'cal'` is the λ-calibrated Voronoi fraction.  In permeation it rests on 3 steady trajectory
  frames, so its profile is noisy (±0.01 per bin) and 3 segments is about what it carries; the mass-fraction profile
  (`'mf'`, every density snapshot) resolves 4–5 segments.  Cell 5 repeats the comparison for all three.
* **Fixed on the way (2026-10-03): the permeation feed baseline.**  In perm_3 the feed piston and the feed face both come
  down below the *reference* gel top; the pore-baseline window then fell back to the reference window, by then in the
  vacuum above the piston, and read 0.  With `P_CAL_MODE = 'pore'` the calibration pressure therefore ran from 1.5 at the
  support *down* to 0.5 at the feed face (it should rise to 1.6), λ was 1.16 instead of 1.09 there, and φ_s^cal came
  out *above* the zero-flux reference on the feed side (0.47–0.49 against 0.447).  The window now follows each
  snapshot's own gel top (`uni.baseline_masks_at`): φ_s^cal runs from 0.33 at the support to 0.44 at the feed face —
  at or just under the reference, as σ′ = 0 there requires (raw Voronoi and mass fraction always did).  The permeation
  notebook's Terzaghi panels used the same baseline and change with it.
* **Reading (run 7 states + perm_3 + the unload of the `_andrelax` run; 2026-10-21).**  Closure (i): the compression curve
  puts σ′ = 0.084 across the fit range for the measured φ_s (0.089 expected for a linear pressure drop): the two decks agree on
  the **equation of state** σ′(φ_s) to ~6 % (the same statement as the M check of the permeation notebook's figure 15).
  Transport, with the unload as the only compression-side D_c: the free re-swelling gives `D_c` = 0.120 at φ_s 0.40
  (κ = D_c/M = 0.32 [0.30, 0.35]), against the steady permeation's local `D_c` ≈ 0.09 (CI 0.06–0.15) at φ_s 0.35–0.40,
  κ_Darcy × M_perm = 0.096 and the Darcy κ = 0.23 [0.20, 0.27]; the permeation transient fits (0.147 / 0.159) sit above the
  local steady values.  Closure (ii): the unload D_c, held constant over the membrane's φ_s range, predicts 1.08× the measured
  flux (1.04 / 1.25 with the Voronoi / mass-fraction φ_s).  The compression and permeation decks therefore agree on transport
  to ~25 % once the hold D_c is left out; the earlier "factor 2" was the hold's 0.05, a load measurement.
* **Opening figures (2026-10-08).**  Figure 1 is `uni.fig_thermo_pressure_norm` on the permeation deck (same code and
  PNG as the permeation notebook's figure 10, lower panel): the two dash-dot guides are `ΔP_th = (4/3)(G/M) ΔP_ext` with
  `G_REF` (0.2, the shear decks) and `G_COMP_REF` (0.05, the compression holds' lateral network stress) and this run's `M`;
  the measured total trace falls between / below them, so which `G` the gel has is read off here.  Figure 2
  (`cmp.fig_solvent_measures`) is the permeation notebook's 7a recast for the membrane community: **concentration**
  `c_s = φ_s^cal ρ_s,0/M_s` (the λ-calibrated Voronoi fraction times the bulk solvent density at zero flux; `M_s` = 1, so
  numerically `c_s = φ_s^cal ρ_s,0` in beads/σ³), the **mass fraction** `ρ_s(z)/ρ_s,0` (= the number fraction: every bead has
  mass 1) and the **probe porosity** `ε_g` (lib/psd.py; PoreBlazer-style, as in Marioni et al.) with `r_probe` = 0.45 σ here
  against the permeation notebook's 0.5 (the probe centre then clears a polymer bead centre by 0.95 σ, about the `U_LJ = kT`
  contact distance of a thermal solvent bead, 0.97 σ).  The three differ for three different reasons: the mass fraction is
  > 1 in the feed reservoir (denser solvent under the feed pressure) and above `φ_s^cal` in the gel (it counts the solvent
  packed against the polymer that the calibration assigns to the network); `ε_g` is below both (the excluded shell around
  every bead counts as solid).  The reservoirs are trimmed to `CONC_MARGIN` σ on either side; the faint curves are the
  individual steady Voronoi frames of `c_s` (its noise), the bands are 95 % over frames / snapshots; panel (b) is the
  bin-by-bin difference of the other two from `φ_s^cal`.
* **Adding a deck.**  Put its Config in `CFG` (the commented `shear` block); `cmp.load_deck` loads it with its own
  library and `DECKS['shear']` is then `dict(kind='shear', cfg, R, levels)` — the same `levels` list the shear notebook
  works on (G, N₁/N₂, the transverse D_c per level).  New cross-deck figures go in `lib/compare.py` as functions of the
  decks they need (as `phi_dependence` takes the compression and permeation decks).